# All Problems

## <u>**Problem 0**</u> 

Create GitHub Issues for the following problems, either using one issue per problem or breaking problems down into appropriate sub-issues.

Use the issues to track your progress. Comment on your progress as you work and close each issue when you consider the corresponding problem complete.

## <u>**Problem 1 - Representing the SHA-256 Data**</u>

#### **1.1 Representing 32-bit Words**

SHA-256 operates on 32-bit words, represented as unsigned integers ranging from $0$ to $2^{32}-1$. In Python, I will use the built-in int type because it supports arithmetic and bitwise operations. However, Python integers have arbitrary precision, so I will use the mask 0xFFFFFFFF to restrict results to 32 bits when necessary. Eight hexadecimal digits represent one 32-bit word.


The following example demonstrates how to represent a 32-bit word
and enforce the 32-bit limit using a bit mask.

In [3]:
import numpy as np

# Define the 32-bit word mask
WORD_MASK = 0xFFFFFFFF

# Represent a 32-bit word
word = 0x12345678

print(f"Word in hexadecimal: {word:08x}")
print(f"Word in decimal: {word}")
print(f"Maximum 32-bit unsigned value: {WORD_MASK}")

# Demonstrate 32-bit arithmetic
result = (WORD_MASK + 1) & WORD_MASK
print(f"Result after 32-bit overflow: {result:08x}")

Word in hexadecimal: 12345678
Word in decimal: 305419896
Maximum 32-bit unsigned value: 4294967295
Result after 32-bit overflow: 00000000


#### Explanation

The value `0x12345678` represents a 32-bit word in hexadecimal notation.
The mask `0xFFFFFFFF` contains 32 one-bits and retains the lowest 32
bits of a value when applied using the bitwise AND operator (`&`).

Python integers can exceed 32 bits, so masking is necessary to reproduce
SHA-256's arithmetic modulo $2^{32}$. As demonstrated, adding one to
the maximum unsigned 32-bit value produces zero after masking.

#### References

1. NIST (2015). *Secure Hash Standard (FIPS PUB 180-4)*,
   Sections 3.1–3.2.
   https://doi.org/10.6028/NIST.FIPS.180-4

2. Python Software Foundation. *Python Documentation: Bitwise
   Operations on Integer Types*.
   https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types

#### **1.2 Representing sequences of 32-bit words**

SHA-256 uses sequences of 32-bit words when processing message blocks and performing its internal calculations. I will represent these sequences using Python lists, which preserve the order of elements and allow individual words to be accessed by index. Each element will be a Python integer restricted to 32 bits using the mask 0xFFFFFFFF when necessary.

A list is appropriate because SHA-256 requires words to be processed in a specific order, and its message schedule contains 64 words.

In [4]:
# Define the 32-bit word mask
WORD_MASK = 0xFFFFFFFF

# Represent a sequence of 32-bit words using a list
words = [
    0x12345678,
    0x9ABCDEF0,
    0x0F1E2D3C,
    0xAABBCCDD
]

# Display the sequence
print("Number of words:", len(words))
print("First word:", f"{words[0]:08X}")
print("Second word:", f"{words[1]:08X}")

# Demonstrate enforcing the 32-bit limit
words.append((0xFFFFFFFF + 1) & WORD_MASK)

print("Last word after masking:", f"{words[-1]:08X}")

Number of words: 4
First word: 12345678
Second word: 9ABCDEF0
Last word after masking: 00000000


#### Explanation

The list preserves the order of the words and allows individual elements to be accessed using indexing. The `len()` function returns the number of elements, while the mask ensures that the calculated result remains within the 32-bit range.

This representation will be useful when storing the 16 words of a message block and the 64 words of the SHA-256 message schedule.

#### References

1. National Institute of Standards and Technology (NIST). (2015).
   *Secure Hash Standard (FIPS PUB 180-4)*, Section 6.2.2.
   https://doi.org/10.6028/NIST.FIPS.180-4

2. Python Software Foundation. *Python 3 Documentation: Data Structures — More on Lists*.
   https://docs.python.org/3/tutorial/datastructures.html

#### **1.3 Representing Input Messages**


SHA-256 processes a message as a sequence of bits. In Python, I will represent input messages using the built-in bytes type, which stores an immutable sequence of byte values from 0 to 255. Text must be encoded into bytes before it can be processed; for this example, I will use UTF-8 encoding. This is consistent with the input processing described in the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4). The [official Python documentation on bytes objects](https://docs.python.org/3/library/stdtypes.html#bytes-objects) explains this representation.



In [5]:
# Define a text message
message = "Hello, SHA-256!"

# Encode the text as UTF-8 bytes
message_bytes = message.encode("utf-8")

# Display the original message and its byte representation
print("Original message:", message)
print("Encoded message:", message_bytes)

# Display the message length in bytes and bits
print("Message length in bytes:", len(message_bytes))
print("Message length in bits:", len(message_bytes) * 8)

Original message: Hello, SHA-256!
Encoded message: b'Hello, SHA-256!'
Message length in bytes: 15
Message length in bits: 120


#### Explanation

The `encode("utf-8")` method converts the text string into a sequence of bytes. The `len()` function returns the number of bytes in that sequence. Since each byte contains eight bits, the 15-byte message has a length of 120 bits.

Using `bytes` provides a suitable representation for SHA-256 input because it supports both encoded text and arbitrary binary data. This allows messages to be processed consistently, regardless of whether they originally represent text or binary content.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*. This standard defines the SHA-256 algorithm and its message-processing requirements. [Available online](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — Bytes Objects*. This documentation explains Python's immutable bytes type and how it represents binary data. [Available online](https://docs.python.org/3/library/stdtypes.html#bytes-objects).

#### **1.4 Representing 512-bit Blocks**

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. I will represent each block using Python's built-in `bytes` type. Each block is divided into sixteen 32-bit words, represented as unsigned integers.

The [NIST Secure Hash Standard (FIPS 180-4), Section 6.2.2](https://doi.org/10.6028/NIST.FIPS.180-4) specifies the SHA-256 message-block processing procedure. Python's [`int.from_bytes()` documentation](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) explains how bytes can be converted into integers using a specified byte order.

The following example demonstrates how to represent a 64-byte block and convert it into sixteen 32-bit words using big-endian byte order.

In [6]:
# Create a 64-byte block containing values from 0 to 63
block = bytes(range(64))

# Display the block length in bytes and bits
print("Block length in bytes:", len(block))
print("Block length in bits:", len(block) * 8)

# Divide the block into sixteen 32-bit words
words = []

for i in range(0, len(block), 4):
    # Convert four bytes into one big-endian 32-bit word
    word = int.from_bytes(block[i:i + 4], byteorder="big")
    words.append(word)

# Display the number of words and the first two words
print("Number of 32-bit words:", len(words))
print("First word:", f"0x{words[0]:08X}")
print("Second word:", f"0x{words[1]:08X}")

Block length in bytes: 64
Block length in bits: 512
Number of 32-bit words: 16
First word: 0x00010203
Second word: 0x04050607


#### Explanation

The expression `bytes(range(64))` creates a sequence of 64 bytes, with values ranging from 0 to 63. Since each byte contains eight bits, the block represents 512 bits.

The loop processes four bytes at a time. The `int.from_bytes()` function combines each group into one 32-bit integer using big-endian byte order, meaning the most significant byte comes first. For example, the bytes `00 01 02 03` become the word `0x00010203`.

The resulting list contains sixteen 32-bit words, which is the representation required for a SHA-256 message block.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 6.2.2. Specifies SHA-256 message-block processing and the use of sixteen 32-bit words. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — int.from_bytes()*. Documents how to convert bytes into integers using a specified byte order. [Available Here](https://docs.python.org/3/library/stdtypes.html#int.from_bytes).

#### **1.5 Representing the Final 256-bit Hash**

SHA-256 produces a 256-bit hash digest, equivalent to 32 bytes. I will represent the digest as a Python `bytes` object because it provides a fixed-length binary representation. For readable output, the digest can be converted into a hexadecimal string containing 64 characters, with each byte represented by two hexadecimal digits.

The [NIST Secure Hash Standard (FIPS 180-4), Section 6.2.2](https://doi.org/10.6028/NIST.FIPS.180-4) specifies that SHA-256 produces a 256-bit message digest. Python's [documentation for bytes objects](https://docs.python.org/3/library/stdtypes.html#bytes-objects) explains this binary representation, and the [`bytes.hex()` documentation](https://docs.python.org/3/library/stdtypes.html#bytes.hex) explains how to convert bytes into hexadecimal text.

The following example demonstrates how to represent a 32-byte digest and display it as a hexadecimal string.

In [7]:
# Create an example 32-byte digest
digest = bytes(range(32))

# Convert the digest to a hexadecimal string
digest_hex = digest.hex()

# Display the digest and verify its length
print("Digest length in bytes:", len(digest))
print("Digest length in bits:", len(digest) * 8)
print("Digest in hexadecimal:", digest_hex)
print("Hexadecimal string length:", len(digest_hex))

Digest length in bytes: 32
Digest length in bits: 256
Digest in hexadecimal: 000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f
Hexadecimal string length: 64


#### Explanation

The expression `bytes(range(32))` creates a sequence of 32 bytes, giving a total length of 256 bits. The `hex()` method converts each byte into two hexadecimal characters, so the resulting string contains 64 characters.

This representation is appropriate because the binary digest can be stored as `bytes`, while its hexadecimal form is convenient for displaying and comparing hash values. In the final implementation, the SHA-256 function will return a 64-character lowercase hexadecimal string.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 6.2.2. Specifies the SHA-256 algorithm and its 256-bit message digest. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — Bytes Objects*. Documents Python's immutable byte sequences and the `bytes.hex()` method for producing hexadecimal representations. [Available Here](https://docs.python.org/3/library/stdtypes.html#bytes-objects).

#### **1.6 Representing big-endian integers**


Big-endian byte order stores the most significant byte first. SHA-256 uses big-endian interpretation when converting message blocks into 32-bit words and when representing integer values in its message-processing steps.

I will use Python's `int.from_bytes()` function to convert a sequence of bytes into an integer, specifying `byteorder="big"`. The `int.to_bytes()` method performs the reverse conversion. These methods are documented in the [official Python documentation for integer methods](https://docs.python.org/3/library/stdtypes.html#int.from_bytes). The [NIST Secure Hash Standard (FIPS 180-4), Sections 3.2 and 6.2.2](https://doi.org/10.6028/NIST.FIPS.180-4) provides the relevant definitions and SHA-256 processing requirements.

The following example demonstrates converting four bytes into a 32-bit integer and converting the integer back into bytes.

In [8]:
# Define four bytes representing a 32-bit integer
byte_data = bytes([0x12, 0x34, 0x56, 0x78])

# Convert the bytes into an integer using big-endian byte order
number = int.from_bytes(byte_data, byteorder="big")

# Convert the integer back into four big-endian bytes
converted_bytes = number.to_bytes(4, byteorder="big")

# Display the original bytes, integer, and converted bytes
print("Original bytes:", byte_data.hex())
print("Integer in hexadecimal:", f"0x{number:08X}")
print("Integer in decimal:", number)
print("Converted bytes:", converted_bytes.hex())

# Demonstrate that the conversion preserves the original bytes
print("Round-trip conversion successful:", converted_bytes == byte_data)

Original bytes: 12345678
Integer in hexadecimal: 0x12345678
Integer in decimal: 305419896
Converted bytes: 12345678
Round-trip conversion successful: True


#### Explanation

The bytes `12 34 56 78` are interpreted in big-endian order, so `0x12` is the most significant byte and `0x78` is the least significant byte. The resulting integer is `0x12345678`, equivalent to `305419896` in decimal.

The `int.from_bytes()` method performs the conversion from bytes to an integer, while `to_bytes()` converts the integer back into a specified number of bytes. The successful round-trip test confirms that the original byte sequence is preserved.

Big-endian conversion is important for SHA-256 because message blocks must be interpreted consistently as 32-bit words. Using an explicit byte order avoids relying on the machine's native byte order.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Sections 3.2 and 6.2.2. Defines the relevant word operations and SHA-256 message-block processing. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- Python Software Foundation. *Built-in Types — Integer Methods*. Documents `int.from_bytes()` and `int.to_bytes()` for conversions between integers and byte sequences. [Available Here](https://docs.python.org/3/library/stdtypes.html#int.from_bytes).

## <u>**Problem 2 - SHA-256 Bitwise Operations**</u>

#### **2.1 Choose Function - Ch(x, y, z)**

The Choose function is one of the six Boolean and bitwise functions used by SHA-256. It selects bits from either `y` or `z`, depending on the corresponding bits in `x`.

It is defined as:

$$
Ch(x, y, z) = (x \land y) \oplus (\neg x \land z)
$$

Where:

- $\land$ represents bitwise AND.
- $\oplus$ represents bitwise XOR.
- $\neg$ represents bitwise NOT.

When a bit in `x` is 1, the corresponding bit from `y` is selected. When a bit in `x` is 0, the corresponding bit from `z` is selected.

The function operates on 32-bit unsigned integers. Its definition is provided in Section 4.1.2 of the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

The function below converts each input to a NumPy `uint32`, so every operation works on 32-bit unsigned values. 
It then applies the Choose definition directly: `x & y` keeps the bits of `y` where `x` is 1, and `~x & z` keeps the bits of `z` where `x` is 0. 
XOR combines the two parts, which never overlap.

In [9]:
def Ch(x, y, z):
    """
    Apply the SHA-256 Choose function to three 32-bit words.

    Args:
        x, y, z: NumPy unsigned 32-bit integers.

    Returns:
        A NumPy unsigned 32-bit integer.
    """
    # Convert inputs to unsigned 32-bit integers
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # Apply the Choose function
    result = (x & y) ^ (~x & z)

    return np.uint32(result)

If every bit of `x` is 1, every bit should be taken from `y`, so the result should equal `y`.

In [10]:
# Define example 32-bit words
x = np.uint32(0xFFFFFFFF)
y = np.uint32(0x12345678)
z = np.uint32(0xABCDEF01)

# Test when x contains all 1 bits
result_1 = Ch(x, y, z)
assert result_1 == y

print("Test 1 (x = all ones):", f"0x{int(result_1):08X}")

Test 1 (x = all ones): 0x12345678


If every bit of `x` is 0, every bit should be taken from `z`, so the result should equal `z`.

In [11]:
# Test when x contains all 0 bits
result_2 = Ch(np.uint32(0), y, z)
assert result_2 == z

print("Test 2 (x = all zeros):", f"0x{int(result_2):08X}")
print("All Choose function tests passed.")

Test 2 (x = all zeros): 0xABCDEF01
All Choose function tests passed.


#### Explanation

The Choose function uses bitwise AND, XOR and NOT operations to select bits from the input words.

The tests confirm both behaviours. Using NumPy's `uint32` type ensures that the operations use 32-bit unsigned values, as required by SHA-256.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 4.1.2. Defines the Choose function used by SHA-256. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- NumPy Developers. *NumPy Data Types*. Documents NumPy's fixed-width unsigned integer types, including `uint32`. [Available Here](https://numpy.org/doc/stable/user/basics.types.html).

#### **2.2 Majority Function — Maj(x, y, z)**


The Majority function is one of the six Boolean and bitwise functions used by SHA-256. It takes three 32-bit words as inputs and determines the majority value of each corresponding bit.

It is defined as:

$$
Maj(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)
$$

Where:

- $\land$ represents bitwise AND.
- $\oplus$ represents bitwise XOR.

For each bit position, the output is 1 when at least two of the three input bits are 1. Otherwise, the output is 0.

The function operates on 32-bit unsigned integers. Its definition is provided in Section 4.1.2 of the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

The function converts each input to a NumPy `uint32` and applies the Majority definition directly. Each AND term is 1 only where a particular pair of inputs are both 1, and XOR combines the three pairs. The result is a bit of 1 wherever at least two of the three inputs have a 1.

In [12]:
def Maj(x, y, z):
    """
    Apply the SHA-256 Majority function to three 32-bit words.

    Args:
        x, y, z: NumPy unsigned 32-bit integers.

    Returns:
        A NumPy unsigned 32-bit integer.
    """
    # Convert inputs to unsigned 32-bit integers
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # Apply the Majority function
    result = (x & y) ^ (x & z) ^ (y & z)

    return np.uint32(result)

If two of the three inputs are all 1 bits, every bit position has a majority of 1s, so the result should be all 1 bits.

In [13]:
# Define example 32-bit words
x = np.uint32(0xFFFFFFFF)
y = np.uint32(0xFFFFFFFF)
z = np.uint32(0x00000000)

# Test when two inputs contain all 1 bits
result_1 = Maj(x, y, z)
assert result_1 == np.uint32(0xFFFFFFFF)

print("Test 1 (two inputs all ones):", f"0x{int(result_1):08X}")

Test 1 (two inputs all ones): 0xFFFFFFFF


If only one input is all 1 bits, every bit position has a majority of 0s, so the result should be zero.

In [14]:
# Test when only one input contains all 1 bits
result_2 = Maj(
    np.uint32(0xFFFFFFFF),
    np.uint32(0x00000000),
    np.uint32(0x00000000)
)
assert result_2 == np.uint32(0x00000000)

print("Test 2 (one input all ones):", f"0x{int(result_2):08X}")

Test 2 (one input all ones): 0x00000000


The inputs `0xAA`, `0xCC` and `0xF0` (repeated across the word) are chosen so that, within each byte, the eight bit positions cover all eight possible combinations of three bits. This makes the test equivalent to checking the function's full truth table. The expected result for each byte is `0xE8`.

In [15]:
# Test with mixed bit patterns
result_3 = Maj(
    np.uint32(0xAAAAAAAA),
    np.uint32(0xCCCCCCCC),
    np.uint32(0xF0F0F0F0)
)
assert result_3 == np.uint32(0xE8E8E8E8)

print("Test 3 (mixed inputs):", f"0x{int(result_3):08X}")
print("All Majority function tests passed.")

Test 3 (mixed inputs): 0xE8E8E8E8
All Majority function tests passed.


#### Explanation

The Majority function uses bitwise AND and XOR operations to calculate the majority value at each bit position.

The expression `(x & y)` identifies positions where both `x` and `y` contain a 1. The other two expressions perform the same operation for the remaining pairs of inputs.

The three results are combined using XOR. This produces a 1 at each position where at least two of the input bits are 1.

The tests demonstrate that the function returns all ones when two inputs contain all ones, returns zero when only one input contains ones, and correctly processes mixed bit patterns.

Using NumPy's `uint32` type ensures that the operations use 32-bit unsigned values, as required by SHA-256.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 4.1.2. Defines the Majority function used by SHA-256. [Available online](https://doi.org/10.6028/NIST.FIPS.180-4).

- NumPy Developers. *NumPy Data Types*. Documents NumPy's fixed-width unsigned integer types, including `uint32`. [Available online](https://numpy.org/doc/stable/user/basics.types.html).

#### **2.3 Sigma0 Function - Σ0(x)**

The upper-case Sigma functions are built from the right rotation operation, defined in Section 3.2 of the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4). A right rotation moves every bit $n$ places to the right, and the bits that fall off the right-hand end wrap around to the left, so no bits are lost.

It is defined as:

$$
ROTR^n(x) = (x \gg n) \lor (x \ll (32 - n))
$$

Where:

- $\gg$ represents a right shift.
- $\ll$ represents a left shift.
- $\lor$ represents bitwise OR.

The function below converts the input to a NumPy `uint32`, shifts it right by $n$ bits, and uses a left shift by $32 - n$ bits to move the lost bits to the top. Because `uint32` has a fixed width, any bits shifted past bit 31 are discarded automatically, so no mask is required.

In [16]:
def ROTR(x, n):
    """Rotate a 32-bit word right by n bits."""
    x = np.uint32(x)

    result = (x >> n) | (x << (32 - n))

    return np.uint32(result)

Rotating `0x00000001` right by one bit should move its only 1 bit from the lowest position to the highest.

In [17]:
# The lowest bit should wrap round to the top
result_1 = ROTR(np.uint32(0x00000001), 1)
assert result_1 == np.uint32(0x80000000)

print("Test 1 (ROTR by 1):", f"0x{int(result_1):08X}")

Test 1 (ROTR by 1): 0x80000000


The Sigma0 function is one of the six Boolean and bitwise functions used by SHA-256. It takes a single 32-bit word and combines three right rotations of that word.

It is defined as:

$$
\Sigma_0^{256}(x) = ROTR^{2}(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)
$$

Where:

- $ROTR^n$ represents a right rotation by $n$ bits.
- $\oplus$ represents bitwise XOR.

The function operates on 32-bit unsigned integers. Its definition is provided in Section 4.1.2 of the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

The function converts the input to a NumPy `uint32`, rotates it by 2, 13 and 22 bits using `ROTR`, and combines the three results using XOR.

In [18]:
def Sigma0(x):
    """Apply the SHA-256 Sigma0 function to a 32-bit word."""
    x = np.uint32(x)

    # XOR of rotations by 2, 13 and 22 bits
    result = ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

    return np.uint32(result)

With only the lowest bit set, each rotation moves that bit to a different position, so the result should contain exactly three 1 bits.

In [19]:
# The single 1 bit should land in three positions
result_2 = Sigma0(np.uint32(0x00000001))
assert result_2 == np.uint32(0x40080400)  

print("Test 2 (Sigma0 of 0x00000001):", f"0x{int(result_2):08X}")
print("All Sigma0 function tests passed.")

Test 2 (Sigma0 of 0x00000001): 0x40080400
All Sigma0 function tests passed.


#### Explanation

The Sigma0 function uses right rotations and XOR to mix bits from three different positions of the input word into each output bit.

The tests confirm that `ROTR` wraps bits around rather than discarding them, and that Sigma0 places the single input bit in three separate positions. Using NumPy's `uint32` type ensures that the operations use 32-bit unsigned values, as required by SHA-256.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Sections 3.2 and 4.1.2. Defines the ROTR operation and the Sigma0 function used by SHA-256. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

- NumPy Developers. *numpy.right_shift* and *numpy.left_shift*. Document NumPy's bitwise shift operations on fixed-width integers. [Available Here](https://numpy.org/doc/stable/reference/generated/numpy.right_shift.html).

#### **2.4 Sigma1 Function - Σ1(x)**

The Sigma1 function is one of the six Boolean and bitwise functions used by SHA-256. Like Sigma0, it takes a single 32-bit word and combines three right rotations of that word, but uses different rotation amounts.

It is defined as:

$$
\Sigma_1^{256}(x) = ROTR^{6}(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)
$$

Where:

- $ROTR^n$ represents a right rotation by $n$ bits.
- $\oplus$ represents bitwise XOR.

The function operates on 32-bit unsigned integers. Its definition is provided in Section 4.1.2 of the [NIST Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

The function reuses the `ROTR` function from 2.3, rotating the input by 6, 11 and 25 bits and combining the three results using XOR.

In [20]:
def Sigma1(x):
    """Apply the SHA-256 Sigma1 function to a 32-bit word."""
    x = np.uint32(x)

    # XOR of rotations by 6, 11 and 25 bits
    result = ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

    return np.uint32(result)

With only the lowest bit set, each rotation moves that bit to a different position, so the result should contain exactly three 1 bits, at positions 26, 21 and 7.

In [21]:
# The single 1 bit should land in three positions
result_1 = Sigma1(np.uint32(0x00000001))
assert result_1 == np.uint32(0x04200080)

print("Test 1 (Sigma1 of 0x00000001):", f"0x{int(result_1):08X}")
print("All Sigma1 function tests passed.")

Test 1 (Sigma1 of 0x00000001): 0x04200080
All Sigma1 function tests passed.


#### Explanation

The Sigma1 function works in the same way as Sigma0, mixing bits from three different positions of the input word into each output bit, but with rotation amounts of 6, 11 and 25.

The test confirms that the single input bit is placed in three separate positions. Using NumPy's `uint32` type ensures that the operations use 32-bit unsigned values, as required by SHA-256.

#### References

- National Institute of Standards and Technology (NIST). *Secure Hash Standard (SHS), FIPS PUB 180-4*, Section 4.1.2. Defines the Sigma1 function used by SHA-256. [Available Here](https://doi.org/10.6028/NIST.FIPS.180-4).

## Problem 4: Padding and Parsing Messages


## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256